# times

Almost every language and database has special functions and/or types for dates, times, and timezones.

For some simple tasks, dates can be stored as strings. For more complex tasks, Python has the builtin `datetime` library, and `pandas` has its own `Timestamp` type. `Timestamp` is designed to work together with `datetime`.

In [ ]:
# This line looks weird, but it's very common
from datetime import datetime, timedelta, UTC

from pathlib import Path
from random import sample

import pandas as pd

In [ ]:
DATA_PATH = Path.cwd() / "data" / "groups.tsv"
print(f"Schedule will be saved here:", DATA_PATH)

## UTC and timezones

**UTC** is a global standard time designed for science and navigation.

UTC is useful for web development and cloud computing.
Web app users might in many countries, and the "local" timezone for a cloud application depends on where it is deployed.
UTC time is always the same everywhere on Earth.

Timezone laws can be confusing. For example, the USA has 4 timezones for the 48 "contiguous" states, plus special rules for "overseas territories," and another timezone for most of Alaska, except some of the Aleutian Islands, which are in the same timezone as Hawaii. Those timezones change twice per year for DST (Daylight Savings Time), except in states and territories which do not use DST.

In [ ]:
# Find your computer's "local" date and time.
# This depends on your computer's timezone settings
# and the local laws in that timezone.
datetime.now()

In [ ]:
# Find UTC time.
datetime.now(UTC)

In [ ]:
# Convert a datetime to string.
str(datetime.now(UTC))

In [ ]:
# Converting strings to datetimes can be confusing.
# There are many different string formats!
# I use the international standard ISO format.
datetime.fromisoformat('2001-01-01 12:34:56.789+00:00')

## pandas.Timestamp

For most of my work, I use `pandas.Timestamp`. I think it's easier to convert strings <-> `Timestamp`, and it comes with extra functions like `date_range`.

In [ ]:
t = pd.Timestamp('1999-12-31 23:59:59.999999999')
print("Tonight we're gonna party like it's")
t

In [ ]:
# For dates, I often use a Timestamp at midnight
pd.Timestamp('2026-04-17')

In [ ]:
# Timestamps can have timezones
pd.Timestamp('2001-01-01 12:34', tz='utc')

In [ ]:
# Current UTC time as a Timestamp
pd.Timestamp.now('utc')

In [ ]:
# Timestamp works with the builtin 'timedelta' type
pd.Timestamp('2025-12-31') + timedelta(days=1)

In [ ]:
# Timestamp can convert itself to datetime.datetime type
pd.Timestamp.now().to_pydatetime()

In [ ]:
# Timestamp can convert itself to datetime.date type
pd.Timestamp.now().date()

In [ ]:
# Timestamp can convert itself to ISO-format strings
pd.Timestamp.now().isoformat(sep=" ", timespec='seconds')

In [ ]:
# The date_range() function creates daily Timestamps
pd.date_range('2026-04-13', '2026-04-17')

In [ ]:
# date_range() can create other equally-spaced ranges
countdown = pd.date_range(
    '1999-12-31 23:59:50',
    '2000-01-01 00:00:00',
    freq='1s'
)
for x in countdown:
    print(x)

## UTC offsets

When we need to show time in a specific timezones, programmers usually find UTC time, then add the **UTC offset** for the timezone.

For example, all of China uses the same UTC offset: +08:00.
Official Chinese clocks are always set to UTC time plus 8 hours.

In [ ]:
t_utc = pd.Timestamp.now('UTC')
print(f"UTC       : {t_utc}")
print(f"Shanghai  : {t_utc.astimezone('Asia/Shanghai')}")
print(f"Hong Kong : {t_utc.astimezone('Asia/Hong_Kong')}")
print(f"Macau     : {t_utc.astimezone('Asia/Macau')}")

## unix time

UNIX time counts the number of seconds since 1970-01-01 midnight UTC.

It's hard to read, but adding and subtracting is easy.
Each moment in time is just a floating-point number.

In [ ]:
def unix_time(t):
    # Convert pandas.Timestamp to UNIX time
    t_zero = pd.Timestamp('1970-01-01')
    return (t - t_zero) / timedelta(seconds=1)

unix_time(pd.Timestamp.now())


In [ ]:
unix_time(pd.Timestamp('1970-01-01 00:01'))

## project presentation times

Choose times for each group to present a project.

In [ ]:
# Class hours start at these times
starts = ["14:00", "14:50", "15:40", "16:30"]
starts = [pd.Timestamp("2026-04-17T" + x) for x in starts]

# Class hours stop at these times
stops = ["14:40", "15:30", "16:20", "17:10"]
stops = [pd.Timestamp("2026-04-17T" + x) for x in stops]

# Check for mistakes
print("Class hours are:")
for x, y in zip(starts, stops):
    print(f"{x} to {y}")

In [ ]:
# For each class hour, create timestamps at 8-minute intervals,
# then create a Series with a DatetimeIndex and string values.
# Concatenate those Series. For now, all values will be null.
hours = zip(starts, stops)
slots = [pd.date_range(x, y, freq='8min') for x, y in hours]
slots = pd.concat(pd.Series(index=x, dtype=str) for x in slots)

# Assign special time slots manually.
slots.iat[0] = "group 0"
slots.loc[stops] = "break"
slots.iat[-2] = "extra time"
slots.iat[-1] = "done"

# Randomly assign groups to empty slots.
groups = sample(range(1, 19), 18)
groups = [f"group {x}" for x in groups]
slots.loc[slots.isnull()] = groups

print(f"Save schedule to {DATA_PATH}")
slots.to_csv(DATA_PATH, sep='\t', header=False)
slots